In [4]:
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
import numpy as np
import pandas as pd

In [12]:
# Charger le modèle
model = tf.keras.models.load_model(
    "../models/attention_lstm_model.keras",
    compile=False
)

# quedarte solo con el head de clasificación
classification_model = Model(
    inputs=model.input,
    outputs=model.get_layer("classification_output").output
)

classification_model.compile(
    loss="binary_crossentropy",
    metrics=["accuracy"]
)



model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20, 5)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 20, 64)    │     17,920 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 20, 64)    │          0 │ lstm[0][0],       │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 20, 64)    │          0 │ lstm[0][0],       │
│                     │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 20, 64)    │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     12,416 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        528 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_re_lu (PReLU)     │ (None, 16)        │         16 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 8)         │        136 │ p_re_lu[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_re_lu_1 (PReLU)   │ (None, 8)         │          8 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ regression_output   │ (None, 1)         │          9 │ p_re_lu_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classification_out… │ (None, 1)         │          9 │ p_re_lu_1[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 31,170 (121.76 KB)

 Trainable params: 31,170 (121.76 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
data = np.load('../data/financial_dataset.npz')

X = data['X']
y_regression = data['y_regression']
y_classification = data['y_classification']
sequence_dates = data['sequence_dates']

In [11]:
train_mask = sequence_dates < np.datetime64('2017-01-01')

val_mask = (
    (sequence_dates >= np.datetime64('2017-01-01')) &
    (sequence_dates < np.datetime64('2018-01-01'))
)

test_mask = sequence_dates >= np.datetime64('2018-01-01')

# TRAIN
X_train = X[train_mask]
y_reg_train = y_regression[train_mask]
y_cls_train = y_classification[train_mask]


# VALIDATION

X_val = X[val_mask]
y_reg_val = y_regression[val_mask]
y_cls_val = y_classification[val_mask]

# TEST

X_test = X[test_mask]
y_reg_test = y_regression[test_mask]
y_cls_test = y_classification[test_mask]

# INFO

print("\n====================")
print("TRAIN")
print("====================")

print(X_train.shape)
print(y_reg_train.shape)
print(y_cls_train.shape)

print("\n====================")
print("VALIDATION")
print("====================")

print(X_val.shape)
print(y_reg_val.shape)
print(y_cls_val.shape)

print("\n====================")
print("TEST")
print("====================")

print(X_test.shape)
print(y_reg_test.shape)
print(y_cls_test.shape)


TRAIN
(460790, 20, 5)
(460790,)
(460790,)

VALIDATION
(123218, 20, 5)
(123218,)
(123218,)

TEST
(12766, 20, 5)
(12766,)
(12766,)


In [14]:
loss, accuracy = classification_model.evaluate(X_train, y_cls_train)

print("Loss:", loss)
print("Accuracy:", accuracy)

14400/14400 ━━━━━━━━━━━━━━━━━━━━ 566s 39ms/step - accuracy: 0.9568 - loss: 0.3222
Loss: 0.32220256328582764
Accuracy: 0.956817626953125


In [ ]:
loss, accuracy = classification_model.evaluate(X_val, y_cls_val)

print("Loss:", loss)
print("Accuracy:", accuracy)

 179/3851 ━━━━━━━━━━━━━━━━━━━━ 2:22 39ms/step - accuracy: 0.9255 - loss: 0.3930

In [13]:
loss, accuracy = classification_model.evaluate(X_test, y_cls_test)

print("Loss:", loss)
print("Accuracy:", accuracy)

2026-05-22 06:53:02.048343: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


399/399 ━━━━━━━━━━━━━━━━━━━━ 16s 39ms/step - accuracy: 0.9749 - loss: 0.3272
Loss: 0.3271879553794861
Accuracy: 0.9748550653457642
